In [1]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml


Set parameter TokenServer to value "sophia1.hpc.ait.dtu.dk"


**Set Up**

In [2]:
fn = 'resources/Nordics_test/networks/base_s_100__12h_2050.nc'


In [3]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.nordics.yaml").read_text())


INFO:pypsa.network.io:New version 1.0.7 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


In [4]:
p = Path(fn)  
try:
    if p.exists():
        p.unlink()
        print(f"Deleted {p}")
    else:
        print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/Nordics_test/networks/base_s_100__12h_2050.nc


**Options**

In [5]:
ongrid=False
cluster_cost_reduction=0
cluster_size=1000   
renewables={"solar",'solar-hsat','onwind'}

In [6]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




In [7]:
nodes_with_clusters

['DE0 0',
 'DE0 1',
 'DE0 10',
 'DE0 11',
 'DE0 12',
 'DE0 13',
 'DE0 14',
 'DE0 15',
 'DE0 16',
 'DE0 17',
 'DE0 18',
 'DE0 19',
 'DE0 2',
 'DE0 20',
 'DE0 21',
 'DE0 22',
 'DE0 23',
 'DE0 24',
 'DE0 25',
 'DE0 26',
 'DE0 27',
 'DE0 28',
 'DE0 29',
 'DE0 3',
 'DE0 30',
 'DE0 31',
 'DE0 32',
 'DE0 33',
 'DE0 34',
 'DE0 35',
 'DE0 36',
 'DE0 37',
 'DE0 38',
 'DE0 39',
 'DE0 4',
 'DE0 40',
 'DE0 5',
 'DE0 6',
 'DE0 7',
 'DE0 8',
 'DE0 9',
 'DK0 0',
 'DK1 0',
 'GB2 0',
 'GB2 1',
 'GB2 10',
 'GB2 11',
 'GB2 12',
 'GB2 13',
 'GB2 14',
 'GB2 15',
 'GB2 16',
 'GB2 17',
 'GB2 18',
 'GB2 19',
 'GB2 2',
 'GB2 20',
 'GB2 21',
 'GB2 22',
 'GB2 23',
 'GB2 24',
 'GB2 25',
 'GB2 3',
 'GB2 4',
 'GB2 5',
 'GB2 6',
 'GB2 7',
 'GB2 8',
 'GB2 9',
 'GB3 0',
 'NL0 0',
 'NL0 1',
 'NL0 2',
 'NL0 3',
 'NL0 4',
 'NL0 5',
 'NL0 6',
 'NL0 7',
 'NL0 8',
 'NO1 0',
 'NO1 1',
 'NO1 2',
 'NO1 3',
 'NO1 4',
 'NO1 5',
 'NO1 6',
 'NO1 7',
 'NO1 8',
 'NO1 9',
 'SE1 0',
 'SE1 1',
 'SE1 10',
 'SE1 2',
 'SE1 3',
 'SE1 4',
 '

**Buses and Generators of the Cluster Addition**

In [11]:
def assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables):
    
    nodes_renewables_cf = {}                #dictionary of dataframes by node and renewable type, sorting the generators by average capacity factor (descending order)
    clusters_generators={}                      #dictionary of dataframes by node and renewable type, containing the generators assigned to the cluster  

    for node in nodes_with_clusters:
        for renewable in renewables:

            nodes_renewables_cf[(node, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )
            print(nodes_renewables_cf[(node, renewable)])

            clusters_generators[(node, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            nodes_renewables_cf[(node, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{node}.*{renewable}$")].mean()
            nodes_renewables_cf[(node, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")]

            nodes_renewables_cf[(node, renewable)] = nodes_renewables_cf[(node, renewable)].sort_values("p_max_pu", ascending=False)

            print(nodes_renewables_cf[(node, renewable)])

            #print(nodes_renewables_cf[(country, renewable)])

            number_gen=0



            while  nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen >= len(nodes_renewables_cf[(node, renewable)]):
                    raise ValueError(f"Not enough {renewable} generators to reach cluster_size.")
                
                number_gen=number_gen+1

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(node, renewable)]  = n.generators.loc[nodes_renewables_cf[(node, renewable)].index[0:number_gen+1]]
            remaining_capacity = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #nodes_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(node, renewable)])

            for idx in clusters_generators[(node, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' cluster'}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(node, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(node, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(node, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(node, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(node, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(node, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(node, renewable)].loc[idx].efficiency,
                    location=clusters_generators[(node, renewable)].loc[idx].location,
                    unit=clusters_generators[(node, renewable)].loc[idx].unit,
                    p_nom_extendable=True,
                    overwrite=True,)


                
                
                n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2 cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node,renewable)].loc[idx].bus + ' H2'}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "substation_off"],
                    )

                ### methanol bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' methanol cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " methanol cluster",
                        v_nom=n.buses.at["EU methanol", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus }", "y"],
                        unit=n.buses.at["EU methanol", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus  }", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus }", "country"],
                        carrier=n.buses.at["EU methanol", "carrier"],
                        control=n.buses.at["EU methanol", "control"],
                        substation_lv=n.buses.at["EU methanol", "substation_lv"],
                        substation_off=n.buses.at["EU methanol", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "substation_off"],
                    )



                if idx == nodes_renewables_cf[(node, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(node, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(node, renewable)].loc[idx].name,
                    )



            

    return n

n= assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables)
            



            

            





        




                   p_max_pu p_nom_max
name                                 
DE0 0 0 solar-hsat      NaN       NaN
                    p_max_pu     p_nom_max
name                                      
DE0 0 0 solar-hsat  0.130178  43706.374841
Remaining top solar-hsat capacity outside the cluster: 42706.3748412665 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
DE0 0 0 solar-hsat  DE0 0      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
DE0 0 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_before  down_time_before  ramp_limit_up  \
name                                                                  
DE0 0 0 sola

Residual capacity of generator DE0 11 0 solar-hsat is name
DE0 11 0 solar-hsat    11889.860628
Name: p_nom_max, dtype: float64 MW
               p_max_pu p_nom_max
name                             
DE0 1 0 solar       NaN       NaN
DE0 1 1 solar       NaN       NaN
DE0 1 2 solar       NaN       NaN
DE0 1 3 solar       NaN       NaN
DE0 1 4 solar       NaN       NaN
DE0 10 0 solar      NaN       NaN
DE0 10 1 solar      NaN       NaN
DE0 10 2 solar      NaN       NaN
DE0 10 3 solar      NaN       NaN
DE0 10 4 solar      NaN       NaN
DE0 11 0 solar      NaN       NaN
DE0 11 1 solar      NaN       NaN
DE0 11 2 solar      NaN       NaN
DE0 11 3 solar      NaN       NaN
DE0 11 4 solar      NaN       NaN
DE0 12 0 solar      NaN       NaN
DE0 12 1 solar      NaN       NaN
DE0 12 2 solar      NaN       NaN
DE0 12 3 solar      NaN       NaN
DE0 12 4 solar      NaN       NaN
DE0 13 0 solar      NaN       NaN
DE0 13 1 solar      NaN       NaN
DE0 13 2 solar      NaN       NaN
DE0 13 3 solar      

Residual capacity of generator DE0 10 2 solar is name
DE0 10 2 solar    260.905411
Name: p_nom_max, dtype: float64 MW
                p_max_pu p_nom_max
name                              
DE0 1 0 onwind       NaN       NaN
DE0 1 1 onwind       NaN       NaN
DE0 1 2 onwind       NaN       NaN
DE0 1 3 onwind       NaN       NaN
DE0 1 4 onwind       NaN       NaN
DE0 10 0 onwind      NaN       NaN
DE0 10 1 onwind      NaN       NaN
DE0 10 2 onwind      NaN       NaN
DE0 10 3 onwind      NaN       NaN
DE0 10 4 onwind      NaN       NaN
DE0 11 0 onwind      NaN       NaN
DE0 11 1 onwind      NaN       NaN
DE0 11 2 onwind      NaN       NaN
DE0 11 3 onwind      NaN       NaN
DE0 11 4 onwind      NaN       NaN
DE0 12 0 onwind      NaN       NaN
DE0 12 1 onwind      NaN       NaN
DE0 12 2 onwind      NaN       NaN
DE0 12 3 onwind      NaN       NaN
DE0 12 4 onwind      NaN       NaN
DE0 13 0 onwind      NaN       NaN
DE0 13 1 onwind      NaN       NaN
DE0 13 2 onwind      NaN       NaN
DE0 13 

Residual capacity of generator DE0 10 1 solar is name
DE0 10 1 solar    27215.232852
Name: p_nom_max, dtype: float64 MW
                p_max_pu p_nom_max
name                              
DE0 10 0 onwind      NaN       NaN
DE0 10 1 onwind      NaN       NaN
DE0 10 2 onwind      NaN       NaN
DE0 10 3 onwind      NaN       NaN
DE0 10 4 onwind      NaN       NaN
                 p_max_pu     p_nom_max
name                                   
DE0 10 4 onwind  0.133797    165.729924
DE0 10 3 onwind  0.105948  11983.845342
DE0 10 2 onwind  0.076256   6605.968230
DE0 10 0 onwind  0.012583   1322.783035
DE0 10 1 onwind  0.000000      0.000000
Remaining top onwind capacity outside the cluster: 11149.575266870452 MW
Capacity of the last onwind generator adjusted to fit cluster size: 834.2700755280814 MW
                    bus control type       p_nom  p_nom_mod  p_nom_extendable  \
name                                                                            
DE0 10 4 onwind  DE0 10      PQ

Residual capacity of generator DE0 12 0 solar-hsat is name
DE0 12 0 solar-hsat    17656.499958
Name: p_nom_max, dtype: float64 MW
               p_max_pu p_nom_max
name                             
DE0 12 0 solar      NaN       NaN
DE0 12 1 solar      NaN       NaN
DE0 12 2 solar      NaN       NaN
DE0 12 3 solar      NaN       NaN
DE0 12 4 solar      NaN       NaN
                p_max_pu    p_nom_max
name                                 
DE0 12 4 solar  0.111231   171.724616
DE0 12 3 solar  0.108370  5038.259707
DE0 12 2 solar  0.105782  6909.592102
DE0 12 1 solar  0.104462  8317.909924
DE0 12 0 solar  0.102281  1040.651300
Remaining top solar capacity outside the cluster: 4209.984323217395 MW
Capacity of the last solar generator adjusted to fit cluster size: 828.2753837426733 MW
                   bus control type       p_nom  p_nom_mod  p_nom_extendable  \
name                                                                           
DE0 12 4 solar  DE0 12      PQ        11.579721

...

/tmp/ipykernel_18245/2144190009.py:93: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/tmp/ipykernel_18245/2144190009.py:93: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/tmp/ipykernel_18245/2144190009.py:93: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/tmp/ipykernel_18245/2144190009.py:93: Perform

Residual capacity of generator NL0 8 2 onwind is name
NL0 8 2 onwind    298.256812
Name: p_nom_max, dtype: float64 MW
                   p_max_pu p_nom_max
name                                 
NO1 0 0 solar-hsat      NaN       NaN
                    p_max_pu     p_nom_max
name                                      
NO1 0 0 solar-hsat  0.120485  31781.802252
Remaining top solar-hsat capacity outside the cluster: 30781.802251635454 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
NO1 0 0 solar-hsat  NO1 0      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
NO1 0 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_before

/tmp/ipykernel_18245/2144190009.py:93: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



Residual capacity of generator NO1 0 3 onwind is name
NO1 0 3 onwind    3842.193922
Name: p_nom_max, dtype: float64 MW
Empty DataFrame
Columns: [p_max_pu, p_nom_max]
Index: []
Empty DataFrame
Columns: [p_max_pu, p_nom_max]
Index: []


ValueError: Not enough solar-hsat generators to reach cluster_size.

In [12]:
n.buses.loc[n.buses.index.str.contains("methanol cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
name,,,,,,,,,,,,,,,,
DE0 0 methanol cluster,1.0,,11.778928,51.997389,methanol,MWh_th,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 11 methanol cluster,1.0,,10.426224,47.834642,methanol,MWh_th,DE0 11,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 10 methanol cluster,1.0,,12.989530,48.436497,methanol,MWh_th,DE0 10,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 19 methanol cluster,1.0,,12.574724,53.967904,methanol,MWh_th,DE0 19,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 16 methanol cluster,1.0,,8.295841,53.275471,methanol,MWh_th,DE0 16,1.0,0.0,inf,PQ,,,DE,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NL0 5 methanol cluster,1.0,,5.788516,51.782718,methanol,MWh_th,NL0 5,1.0,0.0,inf,PQ,,,NL,NaN,NaN
NL0 6 methanol cluster,1.0,,6.400371,52.253520,methanol,MWh_th,NL0 6,1.0,0.0,inf,PQ,,,NL,NaN,NaN
NL0 7 methanol cluster,1.0,,4.187932,52.048726,methanol,MWh_th,NL0 7,1.0,0.0,inf,PQ,,,NL,NaN,NaN


In [13]:
n.generators_t['p_max_pu']

name,DE0 0 0 onwind,DE0 0 0 solar,DE0 0 0 solar rooftop,DE0 0 0 solar-hsat,DE0 0 1 onwind,DE0 0 1 solar,DE0 0 1 solar rooftop,DE0 0 2 onwind,DE0 0 2 solar,DE0 0 2 solar rooftop,...,NL0 8 0 solar-hsat cluster,NL0 8 4 solar cluster,NL0 8 4 onwind cluster,NL0 8 3 onwind cluster,NL0 8 2 onwind cluster,NO1 0 0 solar-hsat cluster,NO1 0 4 solar cluster,NO1 0 3 solar cluster,NO1 0 4 onwind cluster,NO1 0 3 onwind cluster
snapshot,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.741777,0.020643,0.020643,0.032510,0.750684,0.026455,0.026455,0.721260,0.032428,0.032428,...,0.018221,0.022472,0.549369,0.569623,0.596201,0.014350,0.022757,0.021922,0.706259,0.368256
2013-01-01 12:00:00,0.421445,0.018548,0.018548,0.013129,0.446756,0.011506,0.011506,0.423309,0.014435,0.014435,...,0.033831,0.042608,0.624631,0.528180,0.420534,0.008971,0.020818,0.010396,0.201099,0.085414
2013-01-02 00:00:00,0.453966,0.032529,0.032529,0.069007,0.481337,0.034545,0.034545,0.529012,0.054569,0.054569,...,0.089495,0.071992,0.293770,0.254937,0.203339,0.012909,0.011767,0.011465,0.675767,0.309599
2013-01-02 12:00:00,0.440688,0.008941,0.008941,0.019673,0.468406,0.010113,0.010113,0.517474,0.025165,0.025165,...,0.035869,0.047803,0.575891,0.517875,0.440059,0.006932,0.005472,0.007200,0.633523,0.281173
2013-01-03 00:00:00,0.763992,0.011339,0.011339,0.024854,0.801316,0.016118,0.016118,0.763201,0.024184,0.024184,...,0.010673,0.015378,0.719793,0.615593,0.562894,0.002584,0.012989,0.006217,0.982853,0.904390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.352212,0.028215,0.028215,0.020343,0.371327,0.023037,0.023037,0.388126,0.023994,0.023994,...,0.040151,0.049344,0.565847,0.485367,0.437372,0.004073,0.010061,0.012358,0.483384,0.179694
2013-12-30 00:00:00,0.244872,0.140841,0.140841,0.132179,0.246074,0.138753,0.138753,0.243761,0.140353,0.140353,...,0.062447,0.057506,0.946048,0.932963,0.881106,0.003721,0.003262,0.000984,0.712193,0.528867
2013-12-30 12:00:00,0.265229,0.074521,0.074521,0.069523,0.280216,0.072137,0.072137,0.319388,0.074391,0.074391,...,0.018966,0.017309,0.979809,0.961056,0.944532,0.002744,0.000000,0.002058,1.000000,0.999514


In [14]:
n.generators.loc[n.generators.index.str.contains('cluster')]

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,location,unit
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 0 solar-hsat cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 4 solar cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 4 onwind cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,267.204664,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 0 3 onwind cluster,DE0 0 cluster,PQ,,0.0,0.0,True,0.0,732.795336,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
DE0 11 4 solar cluster,DE0 11 cluster,PQ,,0.0,0.0,True,0.0,0.506037,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NO1 0 0 solar-hsat cluster,NO1 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
NO1 0 4 solar cluster,NO1 0 cluster,PQ,,0.0,0.0,True,0.0,764.808856,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
NO1 0 3 solar cluster,NO1 0 cluster,PQ,,0.0,0.0,True,0.0,235.191144,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,


**Links of the Cluster Addition**

In [15]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        ### Methanolization ###

        link_name = f"{node} methanolisation"
        cluster_methanol_bus_name = f"{node} methanol cluster"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=cluster_methanol_bus_name,
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        n.add(
            "Link",
            name=f"{node} methanol cluster",
            bus0=cluster_methanol_bus_name,
            bus1=n.links.at[link_name, "bus1"],
            p_nom_extendable=True,
            carrier=n.buses.loc[n.links.at[link_name, "bus1"], "carrier"],  
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,


        )

    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)



        


        

Index(['DE0 1 H2 Electrolysis cluster'], dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster'], dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster'], dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster'], dtype='object', name='name')
Index(['DE0 1 methanolisation cluster'], dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster',
       'DE0 1 methanol cluster'],
      dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster'], dtype='object', name='name')
Index(['DE0 1 methanolisation cluster'], dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster',
       'DE0 1 methanol cluster'],
      dtype='object', name='name')
Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster'], dtype='object', name='name')
Index(['DE0 1 methanolis

In [16]:
n.links.loc[n.links.index.str.contains("methanolisation cluster")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 methanolisation cluster,DE0 0 H2 cluster,DE0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 1 methanolisation cluster,DE0 1 H2 cluster,DE0 1 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 10 methanolisation cluster,DE0 10 H2 cluster,DE0 10 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 11 methanolisation cluster,DE0 11 H2 cluster,DE0 11 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DE0 12 methanolisation cluster,DE0 12 H2 cluster,DE0 12 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 5 methanolisation cluster,SE1 5 H2 cluster,SE1 5 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 6 methanolisation cluster,SE1 6 H2 cluster,SE1 6 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 7 methanolisation cluster,SE1 7 H2 cluster,SE1 7 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


**Storages of the Cluster Addition**

In [17]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
    return n

n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)





Index(['DE0 1 H2 Electrolysis cluster', 'DE0 1 methanolisation cluster',
       'DE0 1 methanol cluster', 'DE0 2 H2 Electrolysis cluster',
       'DE0 2 methanolisation cluster', 'DE0 2 methanol cluster',
       'DE0 3 H2 Electrolysis cluster', 'DE0 3 methanolisation cluster',
       'DE0 3 methanol cluster', 'GB2 1 H2 Electrolysis cluster',
       'GB2 1 methanolisation cluster', 'GB2 1 methanol cluster',
       'GB2 2 H2 Electrolysis cluster', 'GB2 2 methanolisation cluster',
       'GB2 2 methanol cluster', 'NO1 1 H2 Electrolysis cluster',
       'NO1 1 methanolisation cluster', 'NO1 1 methanol cluster',
       'NO1 2 H2 Electrolysis cluster', 'NO1 2 methanolisation cluster',
       'NO1 2 methanol cluster', 'NO1 3 H2 Electrolysis cluster',
       'NO1 3 methanolisation cluster', 'NO1 3 methanol cluster',
       'NO1 4 H2 Electrolysis cluster', 'NO1 4 methanolisation cluster',
       'NO1 4 methanol cluster', 'NO1 5 H2 Electrolysis cluster',
       'NO1 5 methanolisation cluster', '

In [18]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [19]:
n.links.loc[n.links["bus1"].str.contains('methanol')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 solid biomass biomass-to-methanol,DE0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 solid biomass biomass-to-methanol,DE0 1 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 10 solid biomass biomass-to-methanol,DE0 10 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 11 solid biomass biomass-to-methanol,DE0 11 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 12 solid biomass biomass-to-methanol,DE0 12 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 7 methanol cluster,SE1 7 methanol cluster,EU methanol,,methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 8 methanolisation cluster,SE1 8 H2 cluster,SE1 8 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 8 methanol cluster,SE1 8 methanol cluster,EU methanol,,methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [20]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 H2 Electrolysis,DE0 1,DE0 1 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 10 H2 Electrolysis,DE0 10,DE0 10 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 11 H2 Electrolysis,DE0 11,DE0 11 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 12 H2 Electrolysis,DE0 12,DE0 12 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 5 H2 Electrolysis cluster,SE1 5 cluster,SE1 5 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 6 H2 Electrolysis cluster,SE1 6 cluster,SE1 6 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 7 H2 Electrolysis cluster,SE1 7 cluster,SE1 7 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [21]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Electrolysis,DE0 0,DE0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 H2 Electrolysis,DE0 1,DE0 1 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 10 H2 Electrolysis,DE0 10,DE0 10 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 11 H2 Electrolysis,DE0 11,DE0 11 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 12 H2 Electrolysis,DE0 12,DE0 12 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 5 H2 Electrolysis cluster,SE1 5 cluster,SE1 5 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 6 H2 Electrolysis cluster,SE1 6 cluster,SE1 6 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 7 H2 Electrolysis cluster,SE1 7 cluster,SE1 7 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [22]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
name,,,,,,,,,,,,,,,,
DE0 0 cluster,380.0,,11.778928,51.997389,AC,MWh_el,DE0 0,1.0,0.0,inf,Slack,,,DE,1.0,1.0
DE0 0 H2 cluster,1.0,,11.778928,51.997389,H2,MWh_LHV,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 0 methanol cluster,1.0,,11.778928,51.997389,methanol,MWh_th,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 0 battery cluster,1.0,,11.778928,51.997389,battery,MWh_el,DE0 0,1.0,0.0,inf,PQ,,,DE,NaN,NaN
DE0 11 cluster,380.0,,10.426224,47.834642,AC,MWh_el,DE0 11,1.0,0.0,inf,PQ,,,DE,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NL0 8 battery cluster,1.0,,4.737308,51.809393,battery,MWh_el,NL0 8,1.0,0.0,inf,PQ,,,NL,NaN,NaN
NO1 0 cluster,380.0,,6.268203,59.324070,AC,MWh_el,NO1 0,1.0,0.0,inf,PQ,,,NO,1.0,1.0
NO1 0 H2 cluster,1.0,,6.268203,59.324070,H2,MWh_LHV,NO1 0,1.0,0.0,inf,PQ,,,NO,NaN,NaN


In [23]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 H2 Store cluster,DE0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,89.430064,0.0,True,0,inf,0.0,NaN
DE0 0 battery cluster,DE0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DE0 1 H2 Store cluster,DE0 1 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
DE0 1 battery cluster,DE0 1 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DE0 10 H2 Store cluster,DE0 10 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 7 battery cluster,SE1 7 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
SE1 8 H2 Store cluster,SE1 8 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
SE1 8 battery cluster,SE1 8 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN


In [24]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
relation/13295785-515-DC,NO1 0,GB2 13,,DC,0.964498,True,0,inf,1400.0,0.0,...,0.0,relation/13295785,LINESTRING (-1.5404269162550226 55.14647596191...,1.0,0.983350,,NaN,,False,685.235419
relation/14126301-450-DC,GB2 10,NL0 7,,DC,0.974115,True,0,inf,1000.0,0.0,...,0.0,relation/14126301,LINESTRING (0.7161575436002887 51.440498299145...,1.0,0.977109,,NaN,,False,258.841749
relation/15775538-600-DC,GB2 11,GB2 15,,DC,0.973505,True,0,inf,2250.0,0.0,...,0.0,relation/15775538,LINESTRING (-4.894821189854914 55.718036768358...,1.0,0.892448,,NaN,,False,285.796500
relation/15781671-525-DC,GB2 8,DK0 0,,DC,0.963421,True,0,inf,1400.0,0.0,...,0.0,relation/15781671,LINESTRING (-0.2365005345670103 52.92099253311...,1.0,0.816407,,NaN,,False,733.286547
relation/16213216-525-DC,DE0 24,NO1 0,,DC,0.966595,True,0,inf,1400.0,0.0,...,0.0,relation/16213216,LINESTRING (6.7544309946114405 58.669060406684...,1.0,0.802530,,NaN,,False,591.911880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DC42-reversed,DE0 13,DE0 9,,DC,0.968206,True,2037,inf,0.0,0.0,...,1.0,"{""url"":""https://data.netzausbau.de/2037-2023/N...","LINESTRING (10.5331297 53.5252973, 9.0113444 4...",NaN,0.000000,confirmed,NaN,,True,520.333441
DC42plus-reversed,DE0 22,DE0 9,,DC,0.970511,True,2037,inf,0.0,0.0,...,1.0,"{""url"":""https://data.netzausbau.de/2037-2023/N...","LINESTRING (10.5331297 53.5252973, 9.6151453 4...",NaN,0.000000,confirmed,NaN,,True,418.165996
DC5-reversed,DE0 5,DE0 0,,DC,0.970153,True,2027,inf,0.0,0.0,...,1.0,{url:https://www.netzentwicklungsplan.de/sites...,"LINESTRING (11.6267388 52.2484924, 11.5745421 ...",NaN,0.000000,in_permitting,NaN,,True,434.000757


In [25]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
name,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.0,0.0,True,0.0,inf,NaN,-1.0,...,0.0,0.0,0.0,0.000000,0.0,True,0,inf,0.0,
DE0 0 co2 stored,DE0 0 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
DE0 1 co2 stored,DE0 1 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
DE0 10 co2 stored,DE0 10 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
DE0 11 co2 stored,DE0 11 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 7 battery cluster,SE1 7 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
SE1 8 H2 Store cluster,SE1 8 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,2224.975719,0.0,True,0,inf,0.0,NaN
SE1 8 battery cluster,SE1 8 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN


In [26]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
DE0 0 OCGT methanol,EU methanol,DE0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 1 OCGT methanol,EU methanol,DE0 1,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 10 OCGT methanol,EU methanol,DE0 10,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 11 OCGT methanol,EU methanol,DE0 11,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
DE0 12 OCGT methanol,EU methanol,DE0 12,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 7 OCGT methanol,EU methanol,SE1 7,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
SE1 8 OCGT methanol,EU methanol,SE1 8,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
SE1 9 OCGT methanol,EU methanol,SE1 9,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [27]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
name,,,,,
AC,0.0,#70af1d,AC,inf,0.0
DC,0.0,#8a1caf,DC,inf,0.0
nuclear,0.0,#ff8c00,nuclear,inf,0.0
solar-hsat,0.0,#fdb915,solar-hsat,inf,0.0
offwind-ac,0.0,#6895dd,Offshore Wind (AC),inf,0.0
...,...,...,...,...,...
oil refining,0.0,#e6e6e6,oil refining,inf,0.0
home battery discharger,0.0,#3c5221,home battery discharger,inf,0.0
rural ground heat pump,0.0,#2fb537,rural ground heat pump,inf,0.0


In [28]:
n.global_constraints

,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,,"AC, DC",<=,9.795462e+07,0.0
biomass limit,operational_limit,NaN,,solid biomass,<=,3.278577e+08,0.0
CO2Limit,co2_atmosphere,NaN,,co2_emissions,<=,0.000000e+00,0.0


In [29]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
name,,,,,,,
DE0 0,DE0 0 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 1,DE0 1 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 10,DE0 10 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 11,DE0 11 low voltage,electricity,,0.0,0.0,-1.0,True
DE0 12,DE0 12 low voltage,electricity,,0.0,0.0,-1.0,True
...,...,...,...,...,...,...,...
SE1 7 urban decentral heat,SE1 7 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True
SE1 8 rural heat,SE1 8 rural heat,rural heat,,0.0,0.0,-1.0,True
SE1 8 urban decentral heat,SE1 8 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True


In [30]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
relation/13295785-515-DC,NO1 0,GB2 13,,DC,0.964498,True,0,inf,1400.0,0.0,...,0.0,relation/13295785,LINESTRING (-1.5404269162550226 55.14647596191...,1.0,0.983350,,NaN,,False,685.235419
relation/14126301-450-DC,GB2 10,NL0 7,,DC,0.974115,True,0,inf,1000.0,0.0,...,0.0,relation/14126301,LINESTRING (0.7161575436002887 51.440498299145...,1.0,0.977109,,NaN,,False,258.841749
relation/15775538-600-DC,GB2 11,GB2 15,,DC,0.973505,True,0,inf,2250.0,0.0,...,0.0,relation/15775538,LINESTRING (-4.894821189854914 55.718036768358...,1.0,0.892448,,NaN,,False,285.796500
relation/15781671-525-DC,GB2 8,DK0 0,,DC,0.963421,True,0,inf,1400.0,0.0,...,0.0,relation/15781671,LINESTRING (-0.2365005345670103 52.92099253311...,1.0,0.816407,,NaN,,False,733.286547
relation/16213216-525-DC,DE0 24,NO1 0,,DC,0.966595,True,0,inf,1400.0,0.0,...,0.0,relation/16213216,LINESTRING (6.7544309946114405 58.669060406684...,1.0,0.802530,,NaN,,False,591.911880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SE1 7 battery discharger cluster,SE1 7 battery cluster,SE1 7 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
SE1 8 battery charger cluster,SE1 8 cluster,SE1 8 battery cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
SE1 8 battery discharger cluster,SE1 8 battery cluster,SE1 8 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


In [31]:
n.links.loc[n.links.index.str.contains("cluster"),n.links.columns.str.contains("bus")]

,bus0,bus1,bus4,bus3,bus2
name,,,,,
DE0 0 H2 Electrolysis cluster,DE0 0 cluster,DE0 0 H2 cluster,,,
DE0 0 methanolisation cluster,DE0 0 H2 cluster,DE0 0 methanol cluster,DE0 0 urban central heat,DE0 0 co2 stored,DE0 0 cluster
DE0 0 methanol cluster,DE0 0 methanol cluster,EU methanol,,,
DE0 1 H2 Electrolysis cluster,DE0 1 cluster,DE0 1 H2 cluster,,,
DE0 1 methanolisation cluster,DE0 1 H2 cluster,DE0 1 methanol cluster,DE0 1 urban central heat,DE0 1 co2 stored,DE0 1 cluster
...,...,...,...,...,...
SE1 7 battery discharger cluster,SE1 7 battery cluster,SE1 7 cluster,,,
SE1 8 battery charger cluster,SE1 8 cluster,SE1 8 battery cluster,,,
SE1 8 battery discharger cluster,SE1 8 battery cluster,SE1 8 cluster,,,


In [32]:
print(n.generators.loc[n.generators.index.str.contains("solar")&
    ~n.generators.index.str.contains("solar-hsat") &~n.generators.index.str.contains("solar thermal") &~n.generators.index.str.contains("solar rooftop")])

                                 bus control type        p_nom  p_nom_mod  \
name                                                                        
DE0 0 0 solar                  DE0 0      PQ          4.346652        0.0   
DE0 0 1 solar                  DE0 0      PQ        233.845010        0.0   
DE0 0 2 solar                  DE0 0      PQ       1029.977920        0.0   
DE0 0 3 solar                  DE0 0      PQ       1271.875595        0.0   
DE0 0 4 solar                  DE0 0      PQ        269.082065        0.0   
...                              ...     ...  ...          ...        ...   
NL0 7 3 solar cluster  NL0 7 cluster      PQ          0.000000        0.0   
NL0 7 2 solar cluster  NL0 7 cluster      PQ          0.000000        0.0   
NL0 8 4 solar cluster  NL0 8 cluster      PQ          0.000000        0.0   
NO1 0 4 solar cluster  NO1 0 cluster      PQ          0.000000        0.0   
NO1 0 3 solar cluster  NO1 0 cluster      PQ          0.000000        0.0   

In [33]:
print(n.generators.loc[n.generators.index.str.contains("solar cluster"), n.generators.columns.isin(['p_nom_max','carrier','location','pnom_extendable'])])


                          p_nom_max carrier location
name                                                
DE0 0 4 solar cluster   1000.000000   solar         
DE0 11 4 solar cluster     0.506037   solar         
DE0 11 3 solar cluster   118.699940   solar         
DE0 10 2 solar cluster   260.905411   solar         
DE0 10 1 solar cluster   739.094589   solar         
...                             ...     ...      ...
NL0 7 3 solar cluster    370.290861   solar         
NL0 7 2 solar cluster    371.424325   solar         
NL0 8 4 solar cluster   1000.000000   solar         
NO1 0 4 solar cluster    764.808856   solar         
NO1 0 3 solar cluster    235.191144   solar         

[110 rows x 3 columns]


In [34]:
n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.str.contains("cluster")]

name,DE0 0 0 solar-hsat cluster,DE0 0 4 solar cluster,DE0 0 4 onwind cluster,DE0 0 3 onwind cluster,DE0 11 0 solar-hsat cluster,DE0 11 4 solar cluster,DE0 11 3 solar cluster,DE0 10 2 solar cluster,DE0 19 4 onwind cluster,DE0 16 4 onwind cluster,...,NL0 8 0 solar-hsat cluster,NL0 8 4 solar cluster,NL0 8 4 onwind cluster,NL0 8 3 onwind cluster,NL0 8 2 onwind cluster,NO1 0 0 solar-hsat cluster,NO1 0 4 solar cluster,NO1 0 3 solar cluster,NO1 0 4 onwind cluster,NO1 0 3 onwind cluster
snapshot,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.032510,0.020257,0.745953,0.720592,0.085242,0.120544,0.157375,0.171789,0.999292,0.692934,...,0.018221,0.022472,0.549369,0.569623,0.596201,0.014350,0.022757,0.021922,0.706259,0.368256
2013-01-01 12:00:00,0.013129,0.012368,0.415434,0.418396,0.047798,0.075827,0.080220,0.090211,0.619978,0.785965,...,0.033831,0.042608,0.624631,0.528180,0.420534,0.008971,0.020818,0.010396,0.201099,0.085414
2013-01-02 00:00:00,0.069007,0.116865,0.626378,0.600330,0.115280,0.086783,0.099190,0.054537,0.941442,0.884405,...,0.089495,0.071992,0.293770,0.254937,0.203339,0.012909,0.011767,0.011465,0.675767,0.309599
2013-01-02 12:00:00,0.019673,0.011756,0.535699,0.552085,0.036100,0.059459,0.037807,0.036225,0.992708,0.876311,...,0.035869,0.047803,0.575891,0.517875,0.440059,0.006932,0.005472,0.007200,0.633523,0.281173
2013-01-03 00:00:00,0.024854,0.022084,0.801217,0.787516,0.041355,0.041224,0.054290,0.042639,0.999592,0.999978,...,0.010673,0.015378,0.719793,0.615593,0.562894,0.002584,0.012989,0.006217,0.982853,0.904390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.020343,0.032795,0.448703,0.417988,0.052077,0.011271,0.017797,0.012294,0.950465,0.979491,...,0.040151,0.049344,0.565847,0.485367,0.437372,0.004073,0.010061,0.012358,0.483384,0.179694
2013-12-30 00:00:00,0.132179,0.146524,0.302903,0.262351,0.149781,0.158531,0.118618,0.166995,0.946172,0.942079,...,0.062447,0.057506,0.946048,0.932963,0.881106,0.003721,0.003262,0.000984,0.712193,0.528867
2013-12-30 12:00:00,0.069523,0.071959,0.341143,0.335387,0.096829,0.116195,0.108920,0.096888,0.889163,0.999977,...,0.018966,0.017309,0.979809,0.961056,0.944532,0.002744,0.000000,0.002058,1.000000,0.999514


**Exporting**

In [35]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'resources/Nordics_test/networks/base_s_100__12h_2050.nc contains: links, stores, global_constraints, buses, sub_networks, storage_units, carriers, lines, generators, loads


<xarray.Dataset> Size: 25MB
Dimensions:                               (snapshots: 730, links_i: 7888,
                                           links_t_efficiency_i: 392,
                                           links_t_p_max_pu_i: 200,
                                           stores_i: 1388,
                                           stores_t_e_min_pu_i: 100,
                                           stores_t_e_max_pu_i: 192,
                                           ...
                                           storage_units_i: 72,
                                           storage_units_t_inflow_i: 47,
                                           carriers_i: 120, lines_i: 175,
                                           generators_i: 2955,
                                           generators_t_p_max_pu_i: 2287,
                                           loads_i: 1695, loads_t_p_set_i: 482)
Coordinates: (12/18)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * links_i                               (links_i) object 63kB 'relation/132...
  * links_t_efficiency_i                  (links_t_efficiency_i) object 3kB '...
  * links_t_p_max_pu_i                    (links_t_p_max_pu_i) object 2kB 'DE...
  * stores_i                              (stores_i) object 11kB 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 800B '...
    ...                                    ...
  * carriers_i                            (carriers_i) object 960B 'AC' ... '...
  * lines_i                               (lines_i) object 1kB '0' '1' ... '99'
  * generators_i                          (generators_i) object 24kB 'GB2 1 n...
  * generators_t_p_max_pu_i               (generators_t_p_max_pu_i) object 18kB ...
  * loads_i                               (loads_i) object 14kB 'DE0 0' ... '...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 4kB 'DE0 0...
Data variables: (12/127)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    links_bus0                            (links_i) object 63kB 'NO1 0' ... '...
    links_bus1                            (links_i) object 63kB 'GB2 13' ... ...
    ...                                    ...
    generators_unit                       (generators_i) object 24kB '' ... ''
    generators_t_p_max_pu                 (snapshots, generators_t_p_max_pu_i) float64 13MB ...
    loads_bus                             (loads_i) object 14kB 'DE0 0 low vo...
    loads_carrier                         (loads_i) object 14kB 'electricity'...
    loads_p_set                           (loads_i) float64 14kB 0.0 0.0 ... 0.0
    loads_t_p_set                         (snapshots, loads_t_p_set_i) float64 3MB ...
Attributes:
    network_name:           Unnamed Network
    network_pypsa_version:  1.0.6
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...